In [2]:
import numpy as np
import pandas as pd

In [3]:
reg_numbers = ["REG-101", "REG-102", "REG-103", "REG-104", "REG-105"]
meds = ["Аспирин", "Парацетамол", "Ибупрофен", "Анальгин", "Но-Шпа"]
series_1 = pd.Series(data=meds, index=reg_numbers, dtype="str", name="Лекарство")
manufacturers = ["Фармстандарт", "Озон", "Обновление", "Биосинтез", "Эвалар"]
series_2 = pd.Series(
    data=manufacturers, index=reg_numbers, dtype="str", name="Производитель"
)
prices = [150.50, 90.00, 210.30, 85.40, 320.00]
series_3 = pd.Series(data=prices, index=reg_numbers, dtype="float")
series_4 = series_1 + " (" + series_2 + ")"
max_price = float(series_3.max())
min_price = float(series_3.min())
mean_price = float(series_3.mean())
series_3

REG-101    150.5
REG-102     90.0
REG-103    210.3
REG-104     85.4
REG-105    320.0
dtype: float64

In [4]:
df_1_data = {
    "Форма выпуска": ["Таблетки", "Капсулы", "Таблетки", "Раствор", "Таблетки"],
    "Дозировка": ["500 мг", "200 мг", "400 мг", "500 мг", "40 мг"],
    "Срок годности": ["3 года", "2 года", "5 лет", "3 года", "2 года"],
}
df_1 = pd.DataFrame(data=df_1_data, index=reg_numbers)
df_2 = pd.DataFrame(
    {"Лекарство": series_1, "Производитель": series_2, "Цена": series_3}
)
df_3 = pd.concat([df_2, df_1], axis=1)
df_3.index.tolist()
df_3.columns.tolist()
df_3.values
df_3.head(2)
df_3.iloc[1:4]
df_3["Лекарство"]
df_3["Цена"]
df_3.loc["REG-103", "Лекарство"]
df_3.iloc[3]["Производитель"]


'Биосинтез'

In [5]:
csv_data = """Дата;Открытие;Закрытие;Объем;Изменение
01.10.2026;150,5;152,2;10000;+1.1%
02.10.2026;-; ;н/д;+0.5%
03.10.2026;153,0;151,8;12000;-0.8%
04.10.2026;151,2;154,0;9500;+1.4%
05.10.2026;154,1;155,5;11000;+0.9%
"""
with open("stocks.csv", "w", encoding="utf-8") as f:
    f.write(csv_data)


In [6]:
df_stocks = pd.read_csv(
    "stocks.csv",
    sep=";",
    header=0,
    usecols=["Дата", "Открытие", "Закрытие", "Объем"],
    na_values=["-", " ", "н/д"],
    decimal=",",
)

df_stocks["Дата"] = pd.to_datetime(df_stocks["Дата"], dayfirst=True)
df_stocks["Открытие"] = df_stocks["Открытие"].astype("float64")
df_stocks["Закрытие"] = df_stocks["Закрытие"].astype("float64")
df_stocks["Объем"] = df_stocks["Объем"].astype("float64")
df_stocks.sort_values(by="Объем", ascending=False)
df_stocks["Opening_Rank"] = df_stocks["Открытие"].rank()
df_stocks.describe()
df_stocks["Закрытие"].corr(df_stocks["Объем"])
df_stocks["Закрытие"].cov(df_stocks["Объем"])
df_dropna = df_stocks.dropna(thresh=3)
df_filled = df_stocks.copy()
price_cols = ["Открытие", "Закрытие"]
df_filled[price_cols] = df_filled[price_cols].apply(
    lambda row: row.fillna(row.mean()), axis=1
)
df_filled["Объем"] = df_filled["Объем"].fillna(df_filled["Объем"].mean())
df_past = pd.DataFrame(
    {
        "Дата": pd.to_datetime(["01.10.2025", "02.10.2025"]),
        "Открытие": [130.0, 131.5],
        "Закрытие": [131.0, 132.0],
        "Объем": [8000.0, 8500.0],
    }
)
df_overlap = pd.DataFrame(
    {
        "Дата": pd.to_datetime(["02.10.2026", "06.10.2026"]),
        "Открытие": [np.nan, 156.0],
        "Закрытие": [np.nan, 157.2],
        "Объем": [np.nan, 13000.0],
    }
)
df_other_stock = pd.DataFrame(
    {
        "Дата": df_stocks["Дата"],
        "Открытие_SBER": [250.0, 251.0, 249.5, 253.0, 252.1],
        "Закрытие_SBER": [251.5, 249.0, 252.0, 251.8, 254.0],
    }
)
pd.concat([df_stocks, df_past], ignore_index=True)
pd.merge(df_stocks, df_past, on="Дата", how="outer")
pd.concat([df_stocks, df_overlap]).drop_duplicates(
    subset=["Дата"], keep="first"
)
pd.merge(df_stocks, df_overlap, on="Дата", how="outer").drop_duplicates(
    subset=["Дата"]
)
pd.concat([df_stocks, df_other_stock.drop(columns="Дата")], axis=1)
pd.merge(df_stocks, df_other_stock, on="Дата", how="inner")


,Дата,Открытие,Закрытие,Объем,Opening_Rank,Открытие_SBER,Закрытие_SBER
0,2026-10-01,150.5,152.2,10000.0,1.0,250.0,251.5
1,2026-10-02,NaN,NaN,NaN,NaN,251.0,249.0
2,2026-10-03,153.0,151.8,12000.0,3.0,249.5,252.0
3,2026-10-04,151.2,154.0,9500.0,2.0,253.0,251.8
4,2026-10-05,154.1,155.5,11000.0,4.0,252.1,254.0
